# Description

We evaluate the scalability of CEQL on the previously generated data with growing number of input variables.

In [1]:
# ceql_scalability_experiment.py
#
# Mirrors operon_scalability_experiment.py but trains/evaluates CEQL (ComplexEQL).
# - Runs CEQL runs_per_d times per d
# - Saves per-run results to CSV
# - Plots mean ± std with uniformly spaced x ticks (labels are d)
# - y axis log scale if y_log_scale is True

from __future__ import annotations

import csv
import time
from dataclasses import replace
from pathlib import Path
from typing import Any, Dict, List, Tuple

import h5py
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from config.scalability_config import DataCFG, OperonCFG, CEQL_TRAIN, CEQL
from src.ComplexEQL import ComplexEQL
from src.utils import set_seed, train


def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    return float(np.mean((y_true - y_pred) ** 2))


def _make_seeds() -> Tuple[int, ...]:
    n = int(getattr(OperonCFG, "runs_per_d", 5))
    cfg_seeds = tuple(getattr(OperonCFG, "seeds", ()))
    if len(cfg_seeds) >= n:
        return cfg_seeds[:n]
    return tuple(range(n))


def _print_run_header(Expression: str, d: int, seed: int, run_j: int, n_runs: int, Xtr: np.ndarray, Xte: np.ndarray) -> None:
    print("\n" + "=" * 110)
    print(f"RUN START | Expression={Expression} | d={d} | run={run_j+1}/{n_runs} | seed={seed}")
    print(f"  X_train: {Xtr.shape} | X_test: {Xte.shape}")
    print(
        "  ceql_cfg: "
        f"device={getattr(CEQL_TRAIN,'device','cpu')}, "
        f"batch={int(getattr(CEQL_TRAIN,'train_batch_size',2**14))}, "
        f"lr={float(getattr(CEQL_TRAIN,'lr',1e-3))}, "
        f"phases=[{int(getattr(CEQL_TRAIN,'phase1_epochs',0))},"
        f"{int(getattr(CEQL_TRAIN,'phase2_epochs',0))},"
        f"{int(getattr(CEQL_TRAIN,'phase3_epochs',0))}], "
        f"l1=[{float(getattr(CEQL_TRAIN,'l1_reg_coeff_phase1',0.0))},"
        f"{float(getattr(CEQL_TRAIN,'l1_reg_coeff_phase2',0.0))},"
        f"{float(getattr(CEQL_TRAIN,'l1_reg_coeff_phase3',0.0))}], "
        f"imag=[{float(getattr(CEQL_TRAIN,'imag_w_coeff_phase1',0.0))},"
        f"{float(getattr(CEQL_TRAIN,'imag_w_coeff_phase2',0.0))},"
        f"{float(getattr(CEQL_TRAIN,'imag_w_coeff_phase3',0.0))}]"
    )
    print("=" * 110)


def _aggregate(rows: List[Dict[str, Any]], Expression: str, d_list: List[int]) -> Dict[int, Dict[str, float]]:
    out: Dict[int, Dict[str, float]] = {}
    for d in d_list:
        sub = [r for r in rows if r["Expression"] == Expression and r["d"] == d]
        tr = np.array([r["train_mse"] for r in sub], dtype=np.float64)
        te = np.array([r["test_mse"] for r in sub], dtype=np.float64)
        out[d] = {
            "train_mean": float(tr.mean()),
            "train_std": float(tr.std(ddof=0)),
            "test_mean": float(te.mean()),
            "test_std": float(te.std(ddof=0)),
        }
    return out


def _plot_errorbars_uniform_x(Expression: str, agg: Dict[int, Dict[str, float]], d_list: List[int]) -> None:
    x_pos = np.arange(len(d_list), dtype=np.int32)

    tr_mean = np.array([agg[d]["train_mean"] for d in d_list], dtype=np.float64)
    tr_std  = np.array([agg[d]["train_std"]  for d in d_list], dtype=np.float64)
    te_mean = np.array([agg[d]["test_mean"]  for d in d_list], dtype=np.float64)
    te_std  = np.array([agg[d]["test_std"]   for d in d_list], dtype=np.float64)

    plt.figure()
    plt.errorbar(x_pos, tr_mean, yerr=tr_std, fmt="-o", capsize=4, label="Train MSE (mean ± std)")
    plt.errorbar(x_pos, te_mean, yerr=te_std, fmt="-o", capsize=4, label="Test MSE (mean ± std)")

    if getattr(OperonCFG, "y_log_scale", True):
        plt.yscale("log")

    plt.xlabel(getattr(OperonCFG, "x_label", "Number of Input Variables"))
    plt.ylabel(getattr(OperonCFG, "y_label", "MSE"))
    plt.title(f"CEQL scalability | Expression={Expression} | {len(d_list)} d-values")

    plt.xticks(x_pos, [str(d) for d in d_list])
    plt.legend()
    plt.tight_layout()
    plt.show()


def _write_report_csv(rows: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    fieldnames = [
        "Expression",
        "d",
        "run",
        "seed",
        "train_mse",
        "test_mse",
        "expr",
    ]

    with path.open("w", newline="", encoding="utf-8") as fp:
        w = csv.DictWriter(fp, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in fieldnames})

    print(f"Saved per-run report: {str(path)}")


def _build_scheduler(optimizer: torch.optim.Optimizer):
    if getattr(CEQL_TRAIN, "scheduler", None) == "ReduceLROnPlateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, **getattr(CEQL_TRAIN, "schedulerparams", {})
        )
    if getattr(CEQL_TRAIN, "scheduler", None) is None:
        return None
    raise ValueError(f"Unknown scheduler: {getattr(CEQL_TRAIN,'scheduler',None)!r}")


def _safe_symbolic_expr(model: ComplexEQL, d: int) -> str:
    try:
        syms = [sp.Symbol(f"X{i+1}") for i in range(d)]
        expr = model.get_symbolic_expression(
            syms,
            rounding_decimals=int(getattr(OperonCFG, "sym_decimals", 15)),
            use_imag=False,
        )
        if expr is None:
            return ""
        return str(expr)
    except Exception:
        return ""


def main() -> None:
    t0_all = time.perf_counter()
    rows: List[Dict[str, Any]] = []

    h5_path = str(DataCFG.out_path)
    Expressions: Tuple[str, ...] = tuple(getattr(OperonCFG, "kinds", ())) or ("P3",)

    seeds = _make_seeds()
    n_runs = len(seeds)

    report_csv_path = Path(getattr(OperonCFG, "report_csv_path", "reports/operon_scalability.csv"))
    # default: write a separate CSV for CEQL without touching your config
    report_csv_path = report_csv_path.with_name(report_csv_path.stem.replace("operon", "ceql") + report_csv_path.suffix)

    device_str = str(getattr(CEQL_TRAIN, "device", "cpu"))
    device = torch.device(device_str)

    print(f"Opening HDF5: {h5_path}")
    with h5py.File(h5_path, "r") as f:
        d_list = list(map(int, f.attrs["d_list"]))
        print(f"d_list = {d_list}")
        print(f"Expressions = {list(Expressions)}")
        print(f"seeds (runs) = {list(seeds)} (runs_per_d={n_runs})")
        print(f"device = {device_str}")

        for Expression in Expressions:
            print(f"\n===== Expression: {Expression} =====")
            for d in d_list:
                print(f"Loading data for Expression={Expression}, d={d} ...")

                Xtr = f[f"data/{Expression}/d{d}/train/X"][...].astype(np.float32, copy=False)
                ytr = f[f"data/{Expression}/d{d}/train/y"][...].astype(np.float32, copy=False).reshape(-1)

                Xte = f[f"data/{Expression}/d{d}/test/X"][...].astype(np.float32, copy=False)
                yte = f[f"data/{Expression}/d{d}/test/y"][...].astype(np.float32, copy=False).reshape(-1)

                print(f"Loaded: Xtr={Xtr.shape}, ytr={ytr.shape}, Xte={Xte.shape}, yte={yte.shape}")

                for run_j, seed in enumerate(seeds):
                    _print_run_header(Expression, d, seed, run_j, n_runs, Xtr, Xte)

                    set_seed(int(seed))

                    # IMPORTANT: set CEQL to accept the current dimensionality d
                    CEQL.n_input_fields = int(d)

                    Xtr_t = torch.tensor(Xtr, device=device)
                    ytr_t = torch.tensor(ytr.reshape(-1, 1), device=device)

                    dataset = TensorDataset(Xtr_t, ytr_t)
                    dataloader = DataLoader(
                        dataset,
                        batch_size=int(getattr(CEQL_TRAIN, "train_batch_size", 2**14)),
                        shuffle=True,
                        drop_last=False,
                    )

                    model = ComplexEQL(CEQL).to(device)
                    loss_fn = nn.MSELoss()
                    optimizer = torch.optim.Adam(model.parameters(), lr=float(getattr(CEQL_TRAIN, "lr", 1e-3)))
                    scheduler = _build_scheduler(optimizer)

                    print("train() started ...")
                    model, _loss_traces = train(
                        model=model,
                        dataloader=dataloader,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        cfg=CEQL_TRAIN,
                        device=device,
                        scheduler=scheduler,
                    )
                    print("train() finished")

                    model.eval()
                    with torch.no_grad():
                        yhat_tr = model(Xtr_t).real.squeeze(-1).detach().cpu().numpy().astype(np.float64, copy=False)

                        Xte_t = torch.tensor(Xte, device=device)
                        yhat_te = model(Xte_t).real.squeeze(-1).detach().cpu().numpy().astype(np.float64, copy=False)

                    tr_mse = mse(ytr.astype(np.float64, copy=False), yhat_tr)
                    te_mse = mse(yte.astype(np.float64, copy=False), yhat_te)

                    expr_str = _safe_symbolic_expr(model, d=d)
                    if expr_str:
                        print("expr_str:", expr_str)

                    rows.append(
                        {
                            "Expression": Expression,
                            "d": int(d),
                            "run": int(run_j),
                            "seed": int(seed),
                            "train_mse": float(tr_mse),
                            "test_mse": float(te_mse),
                            "expr": expr_str,
                        }
                    )

                    print(
                        f"RESULT | Expression={Expression} | d={d:>3} | run={run_j+1}/{n_runs} | seed={seed} | "
                        f"train_mse={tr_mse:.6e} | test_mse={te_mse:.6e}"
                    )

    _write_report_csv(rows, report_csv_path)

    print("\n=== Summary (mean±std over runs) ===")
    for Expression in Expressions:
        d_list_sorted = sorted({r["d"] for r in rows if r["Expression"] == Expression})
        agg = _aggregate(rows, Expression=Expression, d_list=d_list_sorted)
        for d in d_list_sorted:
            a = agg[d]
            print(
                f"Expression={Expression} d={d:>3} | "
                f"train_mse={a['train_mean']:.6e}±{a['train_std']:.6e} | "
                f"test_mse={a['test_mean']:.6e}±{a['test_std']:.6e}"
            )
        _plot_errorbars_uniform_x(Expression=Expression, agg=agg, d_list=d_list_sorted)

    total_s = time.perf_counter() - t0_all
    print(f"\nAll done in {total_s:.2f} s")


if __name__ == "__main__":
    main()


Opening HDF5: scalability_experiment_data.h5
d_list = [2, 4, 8, 16, 32, 64]
Expressions = ['P3']
seeds (runs) = [0, 1, 2, 3, 4] (runs_per_d=5)
device = cpu

===== Expression: P3 =====
Loading data for Expression=P3, d=2 ...
Loaded: Xtr=(10000, 2), ytr=(10000,), Xte=(10000, 2), yte=(10000,)

RUN START | Expression=P3 | d=2 | run=1/5 | seed=0
  X_train: (10000, 2) | X_test: (10000, 2)
  ceql_cfg: device=cpu, batch=16384, lr=0.001, phases=[10000,10000,10000], l1=[1e-10,0.001,1e-07], imag=[1e-10,1e-10,0.001]
Random seed set as 0
train() started ...
[PHASE1 | Epoch 1] lr=1.00e-03, total=2.5709e+04, data=2.5709e+04, sparsity_reg=8.1364e-10, imag_w=2.1268e-10, active_edges=33


KeyboardInterrupt: 